In [0]:
%sql
-- === PIPELINE SOURCE CODE (SQL equivalent) ===
-- Attach this notebook to a Lakeflow Pipeline

-- Bronze: stream new CSV files from landing volume (Auto Loader enabled via STREAM keyword)
CREATE OR REFRESH STREAMING TABLE maintenance_bronze_sql
AS SELECT *
FROM STREAM read_files(
    '/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_*.csv',
    format => 'csv',
    header => true
);

-- Silver: apply CDC turbine status changes as SCD Type 2
CREATE OR REFRESH STREAMING TABLE turbines_silver_sql;

CREATE FLOW turbines_cdc_flow
AS AUTO CDC INTO turbines_silver_sql
FROM STREAM(trainer_demo.demo_07.turbine_events)
KEYS (turbine_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY sequence_num
COLUMNS * EXCEPT (operation, sequence_num)
STORED AS SCD TYPE 2;